<table>
<tr><td><img style="height: 150px;" src="images/geo_hydro1.jpg"></td>
<td bgcolor="#FFFFFF">
    <p style="font-size: xx-large; font-weight: 900; line-height: 100%">AG Dynamics of the Earth</p>
    <p style="font-size: large; color: rgba(0,0,0,0.5);">Juypter notebooks</p>
    <p style="font-size: large; color: rgba(0,0,0,0.5);">Georg Kaufmann</p>
    </td>
</tr>
</table>

# Machine learning: Linear regression 

----
*Georg Kaufmann,
Geophysics Section,
Institute of Geological Sciences,
Freie Universität Berlin,
Germany*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

### Create sample data set

We first create a sample data set, using a polynomial function $f(x)$, and shaking both the
$x$ and the $y$ coordinates with uniform random noise:

In [ ]:
def f(x):
    y = x**3
    return y

In [ ]:
n = 21
np.random.seed(1234)
x = np.sort(np.random.uniform(-2,2,n))
#x = np.linspace(-2,2,n)
np.random.seed(1234)
y = f(x) + np.random.uniform(-2,2,n)

print(x.shape,y.shape)

We plot the function. The cubic polynom can still be seen, but the random noise distorts it quite a bit.

In [ ]:
# plot data points
fig=plt.figure(figsize=(12.0, 6.0))
plt.plot(x,y,marker='o',linestyle='none',color='red',label='data')
plt.legend()
plt.show()

Next, we want to use the data set in a **machine learning** algorithm. As we have seen  before,
the **feature matrix** $X$ has to be a two-dimensional array, even in the case with just
**one** feature, the $x$ coordinate.

We therefor need to extend the data array into the 2nd dimension for machine-learning algorithms:

In [ ]:
#X = x[:, np.newaxis]
X=x.reshape(n,1)
print(X.shape)

## Ordinary Least Squares

We want to minimize a residual $\chi^2$, with $y_i$ our **target vector** (the $y$ values) and
$X_{ij}$ our **feature matrix** (the $x$ values), with only one feature ($j=1$):
$$
\chi^2 = \sum\limits_{i=1}^{n} |y_i - X_{ij} w_j|^2
$$
We are looking for the **weights** $w_i=(w_0,w_1,\dots,w_n)$, which satisfy the prediction
$$
p_i(w_j) = w_0 + w_1 x_1 + \dots w_n x_n
$$

In our case
$$
\chi^2 = \sum\limits_{i=1}^{n} |y_i - w_0 - w_1 x_i|^2
$$
with
- $w_0$: intercept,
- $w_1$: slope.

Load the `LinearRegression` module from `scikit_learn`, assign as instance, then call the **fitting part**:
`reg.fit`.

In [ ]:
from sklearn import linear_model
reg = linear_model.LinearRegression()
reg.fit(X,y)

In [ ]:
x_test = np.linspace(-2,2,100)
fig=plt.figure(figsize=(12.0, 6.0))
plt.xlim([-2.1,2.1])
plt.ylim([-10,10])
plt.plot(x_test, f(x_test),linewidth=5,color='gray',label="true")
plt.plot(x_test, reg.predict(x_test[:, np.newaxis]),linewidth=2,label="Model")
plt.plot(x,y,marker='o',linestyle='none',color='red',label='data')
plt.legend()
plt.show()
print ('Score:     ',reg.score(x_test[:, np.newaxis],f(x_test)))
print ('Intercept: ',reg.intercept_)
print ('Slope:     ',reg.coef_)

## Ridge regression

An extension of the **ordinary least square regression** is the **ridge regression**. Here,
minimisation is done over the sum of two terms:
1. least-squares term: $\sum_n |y_i - X_{ij} w_j|^2$
2. regularisation term: $\alpha |w_i|^2$

The resulting residual $\chi^2$, with $y_i$ our **target vector** (the $y$ values) and
$X_{ij}$ our **feature matrix** (the $x$ values), with only one feature ($j=1$), is:
$$
\chi^2 = \sum\limits_{i=1}^{n} |y_i - X_{ij} w_j|^2 + \alpha |w_i|^2
$$
Here, $\alpha$ is a weight for the regularistion term, with $\alpha=0$ reproducing the
least-squares solution.

The regularisation term $|w_i|^2$ will force the feature vector $w_i$ to have a **small length**,
thus the individual coefficients of the feature vector $w_i$ are of similar magnitude.

In [ ]:
from sklearn import linear_model 
ridge = linear_model.Ridge(alpha=0.5)
ridge.fit(X,y)

In [ ]:
x_test = np.linspace(-2,2,100)
fig=plt.figure(figsize=(12.0, 6.0))
plt.xlim([-2.1,2.1])
plt.ylim([-10,10])
plt.plot(x_test, f(x_test),linewidth=5,color='gray',label="true")
plt.plot(x_test, reg.predict(x_test[:, np.newaxis]),linewidth=2,label="Ordinary regression")
plt.plot(x_test, ridge.predict(x_test[:, np.newaxis]),linewidth=2,label="Ridge regression")
plt.plot(x,y,marker='o',linestyle='none',color='red',label='data')
plt.legend()
plt.show()
print ('Score:     ',ridge.score(x_test[:, np.newaxis],f(x_test)))

## Excursion: Scaling features

If the feature matrix includes features with largely varying ranges, is it better to **normalize**
the feature matrix before attempting a fit. This can be done with the `StandardScaler` method from
**preprocessor** library.

The `StandardScaler` calcumates mean $\mu$ and standard deviation $\sigma$ for each column in
the feature matrix $X$, which are then used to transform the feature matrix $\bar{X}$:
$$
\bar{X} = {{X-\mu}\over{\sigma}}
$$
This is done with two steps, first calculating mean and standard deviation with the `fit` method,
then changing the feature matrix with the `transform` method.

As an example, consider a feature matrix $X=[0,1,2,3,4,5,6,7,8,9,10]$.

Note that the `StandardScaler` calculates the variance, $\sigma^2$, but uses standard deviation $\sigma$
to scale...

In [ ]:
from sklearn.preprocessing import StandardScaler
# define feature matrix
a=np.array([0,1,2,3,4,5,6,7,8,9,10])
print('mean:     ',a.mean())
print('variance: ',a.var())
print('std dev:  ',a.std())
# create feature matrix
A=a.reshape(len(a),1)
print(A)
# calculate mean and variance
scaler=StandardScaler()
scaler.fit(A)
print('Scaling parameter: ',scaler.mean_,scaler.var_)
# apply transform to feature matrix
A=scaler.transform(A)
print(A)

## Stochastic gradient descend

text

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import  SGDRegressor
#scaler = StandardScaler()
#scaler.fit(X)
#X = scaler.transform(X)

sgd = SGDRegressor()
sgd.fit(X,y)

In [ ]:
x_test = np.linspace(-2,2,100)
fig=plt.figure(figsize=(12.0, 6.0))
plt.xlim([-2.1,2.1])
plt.ylim([-10,10])
plt.plot(x_test, f(x_test),linewidth=5,color='gray',label="true")
plt.plot(x_test, reg.predict(x_test[:, np.newaxis]),linewidth=2,label="Ordinary regression")
plt.plot(x_test, sgd.predict(x_test[:, np.newaxis]),linewidth=2,label="Stochastic gradient descend")
plt.plot(x,y,marker='o',linestyle='none',color='red',label='data')
plt.legend()
plt.show()
print ('Score:     ',ridge.score(x_test[:, np.newaxis],f(x_test)))

[next>](04_regression2.ipynb)